In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Memory experiment analysis: min-distance + ROC curves.
Refactored into a single organized script. 
Modify only `run_experiment_at_noise` to explore new dynamics.
"""

# ===================== Imports =====================
import sys, os, glob, json, math
from collections import defaultdict

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# project-specific paths
sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('../utls/')
sys.path.append('../src/model/')
sys.path.append("/om2/user/bjmedina/auditory-memory/memory/")

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

import DistanceMemoryModel
import encoders
from utls.plotting import ensure_dir
from utls.loading import load_results_with_exclusion_2, move_sequences_to_used


# ===================== Config =====================
NV_DEFAULT   = 0.2      # per-dim noise std per drift step
SEED         = 123
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
PC_DIMS      = 256
TIMES_TO_PLOT = [10, 17, 22, 40, 80, 119]
NOISE_LEVELS = [0.2, 0.3, 0.5, 0.8, 1.0]


# ===================== Utilities =====================
def roc_from_arrays(hit_scores, fa_scores, score_type="distance"):
    """
    Compute ROC curve and AUC given hit and FA scores.

    Args:
        hit_scores (np.ndarray): scores for hits
        fa_scores  (np.ndarray): scores for false alarms
        score_type (str): 
            "distance"   -> smaller = more similar (flip sign for ROC)
            "likelihood" -> larger = more similar (use directly)

    Returns:
        fpr, tpr, auc_val
    """
    if hit_scores.size == 0 or fa_scores.size == 0:
        return None

    y_true = np.concatenate([
        np.ones_like(hit_scores), 
        np.zeros_like(fa_scores)
    ])
    y_score = np.concatenate([hit_scores, fa_scores])

    if score_type == "distance":
        # lower = better → flip sign so higher = better
        y_score = -y_score
    elif score_type == "likelihood":
        # higher = better → leave as is
        pass
    else:
        raise ValueError(f"Unknown score_type: {score_type}")

    fpr, tpr, _ = roc_curve(y_true, y_score)
    return fpr, tpr, auc(fpr, tpr)

def plot_roc(ax, fpr, tpr, label):
    ax.plot(fpr, tpr, label=label)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)


# ===================== Flexible Runners =====================

def run_experiment_at_noise(
    NV_value,
    *,
    X0, name_to_idx, experiment_list,
    DEVICE="cpu", DistanceMemoryModel=None, zscore_projector=None
):
    """Default runner using DistanceMemoryModel (min distance)."""
    hit_dists, fa_dists = [], []
    isi_hit_dists = defaultdict(list)
    T_max = max((len(seq) for seq in experiment_list), default=0)
    fa_by_t = [[] for _ in range(T_max)]

    for seq in experiment_list:
        if not seq:
            continue

        model = DistanceMemoryModel.DistanceMemoryModel(
            encoding_model=zscore_projector,
            noise_variance=float(NV_value),
            criterion=0.0,
            device=DEVICE
        )
        model.memory_bank = []
        seq_idx = [name_to_idx[fn] for fn in seq]
        bank_set, last_seen = set(), {}

        with torch.no_grad():
            for t, idx_incoming in enumerate(seq_idx, start=1):
                if len(model.memory_bank) > 0:
                    model.apply_noise_to_memory()

                if len(model.memory_bank) > 0:
                    bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)
                    probe = X0[idx_incoming].view(1, -1)
                    dmin = torch.linalg.norm(bank_mat - probe, dim=1).min().item()

                    if idx_incoming in bank_set:
                        hit_dists.append(dmin)
                        isi = t - last_seen[idx_incoming]
                        isi_hit_dists[isi].append((dmin, t))
                    else:
                        fa_dists.append(dmin)
                        fa_by_t[t-1].append(dmin)

                model.memory_bank.append(X0[idx_incoming].detach().clone())
                bank_set.add(idx_incoming)
                last_seen[idx_incoming] = t

    return {
        "hits": np.asarray(hit_dists, float),
        "fas": np.asarray(fa_dists, float),
        "isi_hit_dists": isi_hit_dists,
        "fa_by_t": fa_by_t,
        "T_max": T_max,
        "score_type": "distance"
    }


def run_experiment_scores_decay(
    sigma0,
    *,
    metric="mahalanobis",
    X0=None, name_to_idx=None, experiment_list=None,
    debug=False, **kwargs
):
    """Alternative runner using Mahalanobis or log-likelihood scoring.
    Returns dict in the same format as other runners.
    """
    assert metric in {"mahalanobis", "loglikelihood"}, f"Unknown metric: {metric}"

    hit_scores, fa_scores = [], []
    isi_hit_dists = defaultdict(list)
    T_max = max((len(seq) for seq in experiment_list), default=0)
    fa_by_t = [[] for _ in range(T_max)]
    D = X0.shape[1]

    for seq in experiment_list:
        if len(seq) == 0:
            continue

        seq_idx = [name_to_idx[fn] for fn in seq]
        memory_bank, bank_set, last_seen = [], set(), {}

        for t, idx_incoming in enumerate(seq_idx, start=1):
            probe = X0[idx_incoming].view(1, -1)
            scores = []

            for mem in memory_bank:
                age = t - mem["t_inserted"]
                if age <= 0:
                    continue
                std = max(sigma0 / np.sqrt(age), 1e-6)
                var = std**2

                diff = probe - mem["mu"]
                sqdist = torch.sum(diff**2).item()

                if metric == "mahalanobis":
                    score = (sqdist**0.5) / std
                else:  # loglikelihood
                    score = (
                        -0.5 * D * np.log(2 * np.pi)
                        - D * np.log(std)
                        - 0.5 * (sqdist / var)
                    )
                scores.append(score)

            if scores:
                score_val = min(scores) if metric == "mahalanobis" else max(scores)

                if idx_incoming in bank_set:
                    hit_scores.append(score_val)
                    isi = t - last_seen[idx_incoming]
                    isi_hit_dists[isi].append((score_val, t))
                else:
                    fa_scores.append(score_val)
                    fa_by_t[t-1].append(score_val)

            # Update memory bank
            memory_bank.append({"mu": X0[idx_incoming].detach().clone(), "t_inserted": t})
            bank_set.add(idx_incoming)
            last_seen[idx_incoming] = t

        if debug and memory_bank:
            print("First mem:", memory_bank[0])
            print("Last mem:", memory_bank[-1])
            input()

    return {
        "hits": np.asarray(hit_scores, float),
        "fas": np.asarray(fa_scores, float),
        "isi_hit_dists": isi_hit_dists,
        "fa_by_t": fa_by_t,
        "T_max": T_max,
        "score_type": "likelihood" if metric=="loglikelihood" else "distance"
    }

# ===================== Analysis Helpers =====================
def rocs_across_noise(
    noise_levels,
    *,
    runner,
    X0, name_to_idx, experiment_list,
    DistanceMemoryModel=None, zscore_projector=None, DEVICE="cpu", **kwargs
):
    curves, runs = {}, {}
    for nv in noise_levels:
        run = runner(
            nv,
            X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
            DistanceMemoryModel=DistanceMemoryModel, zscore_projector=zscore_projector,
            DEVICE=DEVICE, **kwargs
        )
        runs[nv] = run

        # 👇 use the runner’s declared score_type instead of guessing
        score_type = run.get("score_type", "distance")
        res = roc_from_arrays(run["hits"], run["fas"], score_type=score_type)

        if res is not None:
            curves[nv] = res
    return curves, runs


def roc_for_isi(run_data, isi_value):
    hits_raw = run_data["isi_hit_dists"].get(isi_value, [])
    if not hits_raw:
        return None

    hits = np.array([d for (d, t) in hits_raw], float)

    # Use all FAs (pooled across time) as comparison set
    fas = np.asarray(run_data["fas"], float)

    score_type = run_data.get("score_type", "distance")
    return roc_from_arrays(hits, fas, score_type=score_type)


def roc_for_second_half(run_data):
    T_half = run_data["T_max"] // 2
    hits, fas = [], []

    for lst in run_data["isi_hit_dists"].values():
        hits.extend([d for (d, t) in lst if t > T_half])
    hits = np.asarray(hits, float)

    for t_idx, bucket in enumerate(run_data["fa_by_t"], start=1):
        if t_idx > T_half:
            fas.extend(bucket)
    fas = np.asarray(fas, float)

    score_type = run_data.get("score_type", "distance")
    return roc_from_arrays(hits, fas, score_type=score_type)
# ===================== Plotting Wrappers =====================
def plot_noise_overlays(curves_by_noise, title="ROC curves across noise levels"):
    fig, ax = plt.subplots(figsize=(6, 6))
    for nv, (fpr, tpr, auc_val) in curves_by_noise.items():
        plot_roc(ax, fpr, tpr, label=f"noise={nv:g} | AUC={auc_val:.3f}")
    ax.plot([0,1], [0,1], 'k--', alpha=0.5)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.2)
    plt.tight_layout(); plt.show()


def plot_isi_and_half_for_noise(nv, run_data, isis=(1, 17)):
    fig, ax = plt.subplots(figsize=(6, 6))

    for k in isis:
        res = roc_for_isi(run_data, k)
        if res is not None:
            fpr, tpr, auc_val = res
            plot_roc(ax, fpr, tpr, label=f"ISI={k} | AUC={auc_val:.3f}")

    res_half = roc_for_second_half(run_data)
    if res_half is not None:
        fpr, tpr, auc_val = res_half
        plot_roc(ax, fpr, tpr, label=f"Second half | AUC={auc_val:.3f}")

    ax.plot([0,1], [0,1], 'k--', alpha=0.5)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"Noise={nv:g}: ISI (0,16) + Second-Half ROCs")
    ax.legend(loc="lower right"); ax.grid(True, alpha=0.2)
    plt.tight_layout(); plt.show

# ===================== Histogram Plotting =====================
def plot_histograms_for_noise(nv, run_data, bins=60, xmax_percentile=99.5):
    """
    Plot histogram of hit vs FA scores for a single noise level.
    Automatically handles 'distance' vs 'likelihood' score types.
    """
    hits = run_data["hits"]
    fas  = run_data["fas"]
    score_type = run_data.get("score_type", "distance")

    pooled = np.concatenate([hits, fas]) if hits.size and fas.size else None
    if pooled is None or pooled.size == 0:
        print(f"[skip] No data for noise={nv}")
        return

    if score_type == "distance":
        xmax = float(np.percentile(pooled, xmax_percentile))
        rng = (0, xmax)
        xlabel = "Min distance"
        title  = f"Distance distributions | noise={nv:g}"
    else:  # likelihood (can be negative)
        xmin = float(np.percentile(pooled, 100 - xmax_percentile))
        xmax = float(np.percentile(pooled, xmax_percentile))
        rng = (xmin, xmax)
        xlabel = "Log likelihood"
        title  = f"Likelihood distributions | noise={nv:g}"

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.hist(hits, bins=bins, range=rng, density=True, alpha=0.55,
            color='green', label=f'hits (N={len(hits)})')
    ax.hist(fas,  bins=bins, range=rng, density=True, alpha=0.55,
            color='red',   label=f'FAs (N={len(fas)})')

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

# ===================== Summary ROCs =====================
def roc_all_isis_across_noise(runs_by_noise):
    pooled_hits, pooled_fas = [], []
    score_type = None
    for run in runs_by_noise.values():
        pooled_hits.extend(run["hits"])
        pooled_fas.extend(run["fas"])
        # ensure consistent score_type
        if score_type is None:
            score_type = run.get("score_type", "distance")
    return roc_from_arrays(np.asarray(pooled_hits), np.asarray(pooled_fas),
                           score_type=score_type)


def roc_non1_isis_across_noise(runs_by_noise):
    pooled_hits, pooled_fas = [], []
    score_type = None
    for run in runs_by_noise.values():
        if "isi_hit_dists" in run:
            for isi, lst in run["isi_hit_dists"].items():
                if isi != 1:
                    pooled_hits.extend([d for (d, _t) in lst])
        else:
            pooled_hits.extend(run["hits"])
        pooled_fas.extend(run["fas"])
        if score_type is None:
            score_type = run.get("score_type", "distance")
    return roc_from_arrays(np.asarray(pooled_hits), np.asarray(pooled_fas),
                           score_type=score_type)


def plot_summary_rocs(runs_by_noise):
    """Plot both summary ROCs side by side."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    # All ISIs
    res_all = roc_all_isis_across_noise(runs_by_noise)
    if res_all is not None:
        fpr, tpr, auc_val = res_all
        plot_roc(axes[0], fpr, tpr, label=f"All ISIs | AUC={auc_val:.3f}")
    axes[0].plot([0,1],[0,1],'k--',alpha=0.5)
    axes[0].set_title("Summary ROC (All ISIs)")
    axes[0].legend(loc="lower right")

    # Non-1 ISIs
    res_non1 = roc_non1_isis_across_noise(runs_by_noise)
    if res_non1 is not None:
        fpr, tpr, auc_val = res_non1
        plot_roc(axes[1], fpr, tpr, label=f"ISI≠1 | AUC={auc_val:.3f}")
    axes[1].plot([0,1],[0,1],'k--',alpha=0.5)
    axes[1].set_title("Summary ROC (Non-1 ISIs)")
    axes[1].legend(loc="lower right")

    for ax in axes:
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

In [ ]:
# ===================== Main Script =====================
# Example: load experiment_list and encode clean reps
# Replace with your own task selection and paths
which_task = "atexts-len120"
seqs_paths = {
    "ind-nature-len120": "mem_exp_ind-nature_2025", 
    "global-music-len120": "global-music-2025-n_80",
    "atexts-len120": "mem_exp_atexts_2025",
    "nhs-region-len120": "nhs-region-n_80"
}
base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/isi_16/len120/"
exps, seqs, fnames = load_results_with_exclusion_2(
    f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
    min_dprime=2, min_trials=120, skip_len60=True, verbose=False, return_skipped=False
)
move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)

# Build experiment_list
experiment_list = []
for seq in seqs:
    with open(base_path.format(seqs_paths[which_task]) + seq, 'r') as f:
        data = json.load(f)
    stim_paths = ["/mindhive/mcdermott/www/mturk_stimuli/bjmedina/" + seqs_paths[which_task] + "/" + s 
                  for s in data['filenames_order']]
    experiment_list.append(stim_paths)

In [ ]:
base_path = "/om/data/public/audioset/wavs/unbalanced_train_segments_downloads/"

# Load CSV
df = pd.read_csv("/om2/user/jmhicks/projects/TextureStreaming/stimuli/OVERLAPPED_0.5s_all_4s_sound_list_with_stationarity_score_no_speech_no_music_audioset_matlab_coch_rms0p02.csv")

# Filter only the rows that pass the stationarity threshold
df["pass_stationarity"] = df["stationarity_score"] <= -0.6

# Group by full filepath
grouped = df.groupby("filepath")["pass_stationarity"]

# Compute fraction of segments that pass per file
fraction_passing = grouped.mean()

# Keep files where more than 50% of segments pass
files_to_use = fraction_passing[fraction_passing > 0.5].index.tolist()
files_to_use = [base_path+f for f in files_to_use]

In [ ]:
# Encode clean reps
pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=PC_DIMS, model_params=model_params,
    sr=20000, rms_level=0.05, duration=2.0, device=DEVICE
)
zscore_projector = encoders.ZScoreSpace(pc_texture_model, device=DEVICE)
zscore_projector.fit(files_to_use[:100])

tmp = DistanceMemoryModel.DistanceMemoryModel(zscore_projector, NV_DEFAULT, criterion=0.0, device=DEVICE)
all_files = sorted({fn for seq in experiment_list for fn in seq})
tmp._fill_memory_bank(all_files)
with torch.no_grad():
    X0 = torch.stack([rep.detach().clone().view(-1) for rep in tmp.memory_bank], dim=0).to(DEVICE)
name_to_idx = {fn: i for i, fn in enumerate(all_files)}

In [ ]:
# Fix sigma0 here
SIGMA0_FIXED = 0.5
NOISE_LEVELS = np.linspace(1, 2.5, 5)  # values for sigma_enc

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    runner=run_experiment_scores_decay,
    X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
    metric="mahalanobis", # or "mahalanobis"
    epochs=1
)

# Per-noise plots
plot_noise_overlays(curves)
for nv in NOISE_LEVELS:
    if nv in runs:
        plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
        plot_histograms_for_noise(nv, runs[nv], bins=60)

# NEW: summary ROC plots
plot_summary_rocs(runs)

In [ ]:
curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    runner=run_experiment_scores_decay,
    X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
    metric="loglikelihood", # or "mahalanobis"
    epochs=1
)

# Per-noise plots
plot_noise_overlays(curves)
for nv in NOISE_LEVELS:
    if nv in runs:
        plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
        plot_histograms_for_noise(nv, runs[nv], bins=60)

# NEW: summary ROC plots
plot_summary_rocs(runs)

In [ ]:
def run_experiment_noisy_decay(
    sigma_enc,
    *,
    sigma0=0.2,          # fixed noise for drift
    epochs=5,
    metric="mahalanobis",
    X0=None, name_to_idx=None, experiment_list=None,
    debug=False, **kwargs
):
    """
    Runner with both drift noise (sigma0) and encoding noise (sigma_enc).
    
    Args:
        sigma_enc (float): encoding noise std (swept across noise_levels)
        sigma0 (float): drift noise std (fixed across runs)
        epochs (int): number of repeated runs
        metric (str): 'mahalanobis' or 'loglikelihood'
    """
    assert metric in {"mahalanobis", "loglikelihood"}, f"Unknown metric: {metric}"

    hit_scores, fa_scores = [], []
    isi_hit_dists = defaultdict(list)
    T_max = max((len(seq) for seq in experiment_list), default=0)
    fa_by_t = [[] for _ in range(T_max)]
    D = X0.shape[1]

    for _ in range(epochs):
        for seq in experiment_list:
            if len(seq) == 0:
                continue

            seq_idx = [name_to_idx[fn] for fn in seq]
            memory_bank, bank_set, last_seen = [], set(), {}

            for t, idx_incoming in enumerate(seq_idx, start=1):
                probe = X0[idx_incoming].view(1, -1)
                scores = []

                # compare probe to all items in memory
                for mem in memory_bank:
                    age = t - mem["t_inserted"]
                    if age <= 0:
                        continue
                    std = max(sigma0 / np.sqrt(age), 1e-6)
                    var = std**2

                    diff = probe - mem["mu"]
                    sqdist = torch.sum(diff**2).item()

                    if metric == "mahalanobis":
                        score = (sqdist**0.5) / std
                    else:  # loglikelihood
                        score = (
                            -0.5 * D * np.log(2 * np.pi)
                            - D * np.log(std)
                            - 0.5 * (sqdist / var)
                        )
                    scores.append(score)

                # classify as hit or FA
                if scores:
                    score_val = min(scores) if metric == "mahalanobis" else max(scores)
                    if idx_incoming in bank_set:
                        hit_scores.append(score_val)
                        isi = t - last_seen[idx_incoming]
                        isi_hit_dists[isi].append((score_val, t))
                    else:
                        fa_scores.append(score_val)
                        fa_by_t[t-1].append(score_val)

                # ---- ENCODING NOISE applied here ----
                noise = torch.randn_like(X0[idx_incoming]) * sigma_enc
                noisy_mu = X0[idx_incoming].detach().clone() + noise

                memory_bank.append({
                    "mu": noisy_mu,
                    "t_inserted": t
                })
                bank_set.add(idx_incoming)
                last_seen[idx_incoming] = t

            if debug and memory_bank:
                print("First mem:", memory_bank[0])
                print("Last mem:", memory_bank[-1])
                input()

    return {
        "hits": np.asarray(hit_scores, float),
        "fas": np.asarray(fa_scores, float),
        "isi_hit_dists": isi_hit_dists,
        "fa_by_t": fa_by_t,
        "T_max": T_max,
        "score_type": "likelihood" if metric == "loglikelihood" else "distance"
    }

# Fix sigma0 here
SIGMA0_FIXED = 0.5
NOISE_LEVELS = np.linspace(1, 2.5, 5)  # values for sigma_enc

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    runner=run_experiment_noisy_decay,
    X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
    sigma0=SIGMA0_FIXED,    # fixed
    metric="loglikelihood", # or "mahalanobis"
    epochs=1
)

In [ ]:
# Per-noise plots
plot_noise_overlays(curves)
for nv in NOISE_LEVELS:
    if nv in runs:
        plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
        plot_histograms_for_noise(nv, runs[nv], bins=60)

# NEW: summary ROC plots
plot_summary_rocs(runs)

In [ ]:
# Fix sigma0 here
SIGMA0_FIXED = 0.5
NOISE_LEVELS = np.linspace(1, 2.5, 5)  # values for sigma_enc

curves, runs = rocs_across_noise(
    NOISE_LEVELS,
    runner=run_experiment_noisy_decay,
    X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
    sigma0=SIGMA0_FIXED,    # fixed
    metric="mahalanobis", # or "mahalanobis"
    epochs=1
)

# Per-noise plots
plot_noise_overlays(curves)
for nv in NOISE_LEVELS:
    if nv in runs:
        plot_isi_and_half_for_noise(nv, runs[nv], isis=(1,17))
        plot_histograms_for_noise(nv, runs[nv], bins=60)

# NEW: summary ROC plots
plot_summary_rocs(runs)